In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, trim, row_number, current_date, current_timestamp, sha2, concat_ws
from pyspark.sql.window import Window
import urllib

# ==========================================
# 1. PARAMETERIZATION & CONFIGURATION
# ==========================================
# Senior engineers NEVER hardcode paths or variables. We use widgets.
dbutils.widgets.text("company_key", "ABC", "Company Key")
dbutils.widgets.text("brand_key", "ABC", "Brand Key")
dbutils.widgets.text("catalog_name", "erp_lakehouse", "Catalog Name")
dbutils.widgets.text("schema_name", "silver", "Schema Name")
dbutils.widgets.text("silver_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/sl_invoiceline", "Silver External Path")
dbutils.widgets.text("exception_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/exception_table", "Exception Table Path")


COMPANY_KEY = dbutils.widgets.get("company_key")
BRAND_KEY = dbutils.widgets.get("brand_key")
CATALOG_NAME= dbutils.widgets.get("catalog_name")
SILVER_SCHEMA= dbutils.widgets.get("schema_name")
SILVER_PATH = dbutils.widgets.get("silver_external_path")
EXCEPTION_PATH = dbutils.widgets.get("exception_external_path")



# Processing SalesInvoicelines table

In [0]:
# ==========================================
# 2. DATA LOADING (BRONZE LAYER)
# ==========================================
print("Reading data from Bronze Delta Table...")
bronze_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_salesinvoicelines")

# Inject control columns early
enriched_df = bronze_df \
    .withColumn('RecordStatus', lit('0')) \
    .withColumn('CompanyKey', lit(COMPANY_KEY)) \
    .withColumn('BrandKey', lit(BRAND_KEY))
filtered_df = enriched_df.filter(enriched_df.lineType != 'Comment')
display(filtered_df.limit(10))


In [0]:
from pyspark.sql.functions import col, when, trim, row_number
from pyspark.sql.window import Window

# ==========================================
# 3. DATA QUALITY & INTEGRITY RULES (SILVER AUDIT)
# ==========================================

print("Executing pre-shuffle Validation Rule 1: Multi-Column Null/Blank Check... 🔍")

# If ANY of these key identifiers are missing, flag it as '2' straight away.
validated_df = filtered_df.withColumn(
    "RecordStatus",
    when(
        (col("id").isNull()) | (trim(col("id")) == "") |
        (col("documentId").isNull()) | (trim(col("documentId")) == ""),
        lit('2')
    ).otherwise(col("RecordStatus"))
)


print("Executing Validation Rule 2: Performance-Optimized Deduplication... 🚀")

# Never order by the ID itself; order by a timestamp to ensure you keep the LATEST operational record.
window_spec = Window.partitionBy("id","documentId", "CompanyKey", "BrandKey").orderBy(col("ingestion_time").desc())

# Apply the window ranking function defensively
processed_df = validated_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn(
        "RecordStatus", 
        # CRITICAL CONTEXT: Only flag as a duplicate ('1') if it passed the previous check and is still '0'
        when((col("RecordStatus") == '0') & (col("row_num") > 1), lit('1')).otherwise(col("RecordStatus"))
    ) \
    .drop("row_num")


# Persist to memory to freeze execution graph before splitting into forks (Exceptions vs Clean Data)
processed_df.cache()

print("Data quality auditing complete. Frame cached successfully.")
display(processed_df.limit(10))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, trim, array, explode, when, current_date, current_timestamp, sha2, concat_ws, coalesce
from pyspark.sql.utils import AnalysisException

print("Processing and routing complex multi-column data exceptions... 🚀")

# Ensure the dedicated silver schema isolation layer exists natively
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Ensure your global exception storage structure is instantiated
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table (
    ExceptionID STRING,
    BrandKey STRING,
    RecordKey STRING,
    TableName STRING,
    ColumnName STRING,
    ExceptionDetails STRING,
    SysCreatedDate DATE,
    SysCreatedBy STRING
)
USING DELTA
LOCATION '{EXCEPTION_PATH}'
""")

# ==========================================
# 1. FORK A: PROCESS DUPLICATE RECORDS (Status '1')
# ==========================================
duplicate_records_df = processed_df.filter(col("RecordStatus") == '1')

duplicate_exceptions = duplicate_records_df.select(
    # Stable, deterministic cryptographic unique identifier
    sha2(concat_ws("||", col("id"),col("documentid"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    # Safely handle potential nulls in ID during string mapping
    coalesce(col("id").cast("string"), lit("UNKNOWN_ID")).alias("RecordKey"),
    lit("bz_SalesInvoicelines").alias("TableName"),
    lit("").alias("ColumnName"),
    lit("Duplicate record identified during delta ranking window").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
)

# ==========================================
# 2. FORK B: PROCESS NULL/BLANK RECORDS (Status '2')
# ==========================================
null_blank_records_df = processed_df.filter(col("RecordStatus") == '2')

# Dynamically construct array elements for validation tracking
columns_to_check = ["id","documentId"]
exprs = [
    when((col(c).isNull()) | (trim(col(c)) == ""), lit(c)).otherwise(lit(None))
    for c in columns_to_check
]

# Pack errors into an array structure
flagged_nulls_df = null_blank_records_df.withColumn("NullColumns", array(*exprs))

# Explode elements to unpack one audit log entry per failing column
null_exceptions = flagged_nulls_df.select(
    sha2(concat_ws("||", col("id"), col("documentid"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    # Senior Strategy: If id is null, fall back to showing the invoice Number so business can search it!
    coalesce(col("id").cast("string"), col("documentid").cast("string"), lit("UNKNOWN_ROW")).alias("RecordKey"),
    lit("bz_SalesInvoicelines").alias("TableName"),
    explode(col("NullColumns")).alias("ColumnName"),
    lit("Null/Blank primary value restriction violation").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
).filter(col("ColumnName").isNotNull()) # Drop rows for columns that actually passed checks

# ==========================================
# 3. CONSOLIDATE AND APPEND TO AUDIT STORE
# ==========================================
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)

# Cache checkpoint to optimize action evaluation
final_exceptions_df.cache()

if final_exceptions_df.count() > 0:
    print(f"Writing exceptions to log repository: {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table")
    # Write cleanly using modern Delta formatting API
    final_exceptions_df.write.format("delta").mode("append").saveAsTable("erp_lakehouse.silver.exception_table")

# Remove the exceptions from your main pipeline dataframe to continue clean processing downstream
clean_result_df = processed_df.filter(~col("RecordStatus").isin(['1', '2']))

# Clear exception tracking from cluster memory cache overhead
final_exceptions_df.unpersist()

print("Exception processing routine complete. Clean corporate records isolated successfully! 🏁")

In [0]:
from pyspark.sql.functions import lit, col, when, datediff, coalesce

# 1. Pre-calculate reusable math expressions to avoid repetitive code blocks
qty = coalesce(col("quantity"), lit(0.0))
net_amt = coalesce(col("NetAmount"), lit(0.0))
disc_amt = coalesce(col("DiscountAmount"), lit(0.0))

# Gross Unit Price: (Net + Discount) / Quantity
gross_unit_price = when(qty == 0, lit(0.0)).otherwise((net_amt + disc_amt) / qty)

# Net Unit Price (Discounted): Net / Quantity
net_unit_price = when(qty == 0, lit(0.0)).otherwise(net_amt / qty)

# 2. Build the optimized selection mapping
final_df = clean_result_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("documentId").alias("InvSrcId").cast("string"),
    col("id").alias("InvLnSrcId").cast("string"),
    
    # Clean source references with coalesce
    coalesce(col("sequence"), lit("")).alias("SOLnSrcId").cast("string"),
    coalesce(col("locationId"), lit("")).alias("WHSrcId").cast("string"),
    coalesce(col("itemid"), lit("")).alias("ItemSrcId").cast("string"),
    coalesce(col("sequence"), lit("")).alias("InvLnKey").cast("string"),
    
    # Conditional type mappings
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED', 'ADM'), lit("Charge"))
        .otherwise(lit("Inv")).alias("InvLnType").cast("string"),
    
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED'), lit("Freight"))
        .when(col("lineObjectNumber") == 'ADM', lit("AddlChrg"))
        .otherwise(lit("Invoice")).alias("InvLnTypeDesc").cast("string"),
    
    # Date handling
    when(col("shipmentDate") == '1900-01-01', lit(None)).otherwise(col("shipmentDate")).alias("ShipDt").cast("timestamp"),
    
    # Item and structural details
    coalesce(col("description"), lit("")).alias("ItemDesc").cast("string"),
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED'), lit("Freight"))
        .when(col("lineObjectNumber") == 'ADM', lit("AddlChrg"))
        .otherwise(lit("Item")).alias("ItemType").cast("string"),
    
    coalesce(col("UnitOfMeasureID"), lit("")).alias("UOMSrcId").cast("string"),
    coalesce(col("unitofMeasureCode"), lit("")).alias("UOM").cast("string"),
    
    # Currency and constant metrics
    lit("INR").alias("CustCurr").cast("string"),
    lit("INR").alias("LocCurr").cast("string"),
    lit(1.0).alias("ExRtLCCC").cast("double"),
    lit(1.0).alias("ExRtCCLC").cast("double"),
    lit(1.0).alias("ExRtLCIN").cast("double"),
    
    # Quantity and Base Cost metrics
    qty.alias("InvLnQty").cast("double"),
    (gross_unit_price*lit(0.75)).alias("UnitCostCC").cast("double"),
    (gross_unit_price*lit(0.75)).alias("UnitCostLC").cast("double"),
    (gross_unit_price*lit(0.75)).alias("UnitCostIN").cast("double"),
    
    # Reusing calculated pricing columns smoothly
    gross_unit_price.alias("UnitPriceCC").cast("double"),
    gross_unit_price.alias("UnitPriceLC").cast("double"),
    gross_unit_price.alias("UnitPriceIN").cast("double"),
    
    net_unit_price.alias("DiscUnitPriceCC").cast("double"),
    net_unit_price.alias("DiscUnitPriceLC").cast("double"),
    net_unit_price.alias("DiscUnitPriceIN").cast("double"),
    
    # Line total costs and values
    (net_amt*lit(0.75)).alias("InvLnCostCC").cast("double"),
    (net_amt*lit(0.75)).alias("InvLnCostLC").cast("double"),
    (net_amt*lit(0.75)).alias("InvLnCostIN").cast("double"),
    
    net_amt.alias("InvLnValCC").cast("double"),
    net_amt.alias("InvLnValLC").cast("double"),
    net_amt.alias("InvLnValIN").cast("double"),
    
    # Discounts and Tax fields
    coalesce(col("DiscountPercent"), lit(0.0)).alias("InvLnDiscPerc").cast("double"),
    disc_amt.alias("InvLnDiscValCC").cast("double"),
    disc_amt.alias("InvLnDiscValLC").cast("double"),
    disc_amt.alias("InvLnDiscValIN").cast("double"),
    
    coalesce(col("TaxPercent"), lit(0.0)).alias("InvLnTaxPerc").cast("double"),
    coalesce(col("NetTaxAmount"), lit(0.0)).alias("InvLnTaxValCC").cast("double"),
    coalesce(col("NetTaxAmount"), lit(0.0)).alias("InvLnTaxValLC").cast("double"),
    coalesce(col("NetTaxAmount"), lit(0.0)).alias("InvLnTaxValIN").cast("double"),
    
    # System timestamps and tracking metadata
    lit(None).alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)

display(final_df.limit(10))

In [0]:
from pyspark.sql.functions import col, lit, when, coalesce

# 1. Alias all DataFrames immediately to keep references clean and unambiguous
f_df  = final_df.alias("f")
inv_df = spark.read.table("erp_lakehouse.silver.sl_invoiceheader").alias("inv")
item_df = spark.read.table("erp_lakehouse.silver.sl_item").alias("item")
prod_df = spark.read.table("erp_lakehouse.silver.sl_productline").alias("prod")

# 2. Chain all left joins in a single execution pass
combined_joins_df = f_df \
    .join(
        inv_df,
        (col("f.CompanyKey") == col("inv.CompanyKey")) & 
        (col("f.BrandKey") == col("inv.BrandKey")) & 
        (col("f.InvSrcId") == col("inv.InvSrcId")),
        "left"
    ) \
    .join(
        item_df,
        (col("f.CompanyKey") == col("item.CompanyKey")) & 
        (col("f.BrandKey") == col("item.BrandKey")) & 
        (col("f.ItemSrcId") == col("item.ItemSrcID")), 
        "left"
    ) \
    .join(
        prod_df,
        (col("f.CompanyKey") == col("prod.CompanyKey")) & 
        (col("f.BrandKey") == col("prod.BrandKey")) & 
        (col("item.ProductLnSrcId") == col("prod.ProductLnSrcId")),
        "left"
    )

# 3. Pull everything together with proactive Null Handling
Invoices_final_df = combined_joins_df.select(
    # Pull all original attributes from your processed final_df base (already null-handled)
    col("f.*"),
    
    # --- Enrichments from Invoice Header ---
    # Safe timestamp fallback
    coalesce(col("inv.ExDt"), lit(None).cast("timestamp")).alias("ExDt"),
    # Business keys fall back to clean empty strings or a standard default description
    coalesce(col("inv.SOKey"), lit("")).alias("SOSrcId").cast("string"),
    coalesce(col("inv.InvKey"), lit("")).alias("InvKey").cast("string"),
    coalesce(col("inv.InvStatus"), lit("Draft")).alias("InvLnStatus").cast("string"),
    
    # --- Enrichments from Item Master ---
    coalesce(col("item.ProductLnKey"), lit("")).alias("ProductLnKey").cast("string"),
    coalesce(col("item.ItemKey"), lit("")).alias("ItemKey").cast("string"),
    coalesce(col("item.ProductLnSrcId"), lit("")).alias("ProductLnSrcId").cast("string"),
    
    # --- Enrichments from Product Line Master ---
    coalesce(col("prod.ProductLnDesc"), lit("")).alias("ProductLnDesc").cast("string"),
    coalesce(col("prod.ProductLnType"), lit("")).alias("ProductLnType").cast("string")
)

# 4. Verify the integrated final output
display(Invoices_final_df.limit(10))


## Processing Creditmemolines

In [0]:

credit_memos_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_salescreditmemolines")

# Initialize structural routing constants
added_df = credit_memos_df.withColumn('RecordStatus', lit('0')) \
                          .withColumn('CompanyKey', lit('ABC')) \
                          .withColumn('BrandKey', lit(BRAND_KEY))



In [0]:
# ==========================================
# 2. DEDUPLICATION & VALIDATION ROUTINE
# ==========================================
# Add row number partitioned by id, documentId, companykey, and brandkey
window_spec = Window.partitionBy("id", "documentId", "CompanyKey", "BrandKey").orderBy("id", "documentId")

result_df = added_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("RecordStatus", when(col("row_num") > 1, lit('1')).otherwise(col("RecordStatus"))) \
    .drop("row_num")

# Identify row-level missing or blank constraints for 'id' or 'documentId'
result_df = result_df.withColumn(
    "RecordStatus",
    when(
        ((col("id").isNull()) | (trim(col("id")) == "")) |
        ((col("documentId").isNull()) | (trim(col("documentId")) == "")),
        lit('2')
    ).otherwise(col("RecordStatus"))
)



In [0]:
# ==========================================
# 3. ENTERPRISE EXCEPTION HANDLING ROUTINE
# ==========================================
duplicate_records_df = result_df.filter(col("RecordStatus") == '1')
null_blank_records_df = result_df.filter(col("RecordStatus") == '2')

# Fork A: Duplicate Log Generation (Using deterministic SHA2 keys instead of monotonically_increasing_id)
duplicate_exceptions = duplicate_records_df.select(
    sha2(concat_ws("||", col("id"), col("documentId"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    concat_ws(" ", col("id"), col("documentId")).alias("RecordKey"),
    lit("bz_SalesCreditLines").alias("TableName"),
    lit("").alias("ColumnName"),
    lit("Duplicate record found during delta ranking window").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("system").alias("SysCreatedBy")
)

# Fork B: Null Constraint Log Generation (Dynamically unpivoting failed attributes via array explode)
columns_to_check = ["id", "documentId"]
exprs = [
    when((col(c).isNull()) | (trim(col(c)) == ""), lit(c)).otherwise(lit(None))
    for c in columns_to_check
]

null_blank_records_df = null_blank_records_df.withColumn("NullColumns", array(*exprs))

null_exceptions = null_blank_records_df.select(
    sha2(concat_ws("||", col("documentId"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    coalesce(col("id").cast("string"), col("documentId").cast("string"), lit("UNKNOWN_ROW")).alias("RecordKey"),
    lit("bz_SalesCreditLines").alias("TableName"),
    explode(col("NullColumns")).alias("ColumnName"),
    lit("Null/ Blanks values").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("system").alias("SysCreatedBy")
).filter(col("ColumnName").isNotNull())

# Union and append exceptions to your isolated logging store
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)
final_exceptions_df.cache()

if final_exceptions_df.count() > 0:
    final_exceptions_df.write.mode("append").insertInto("erp_lakehouse.silver.exception_table")

final_exceptions_df.unpersist()

# Filter out exception records from the main execution branch
clean_result_df = result_df.filter(~col("RecordStatus").isin(['1', '2']))



In [0]:
# ==========================================
# 4. COLUMN MAPPING & FINANCIAL BALANCING
# ==========================================
# Pre-calculate reusable math variables for credit-memo reversals (-1 balancing logic)
qty = coalesce(col("quantity"), lit(0.0))
inv_qty = qty * -1.0
net_amt = coalesce(col("NetAmount"), lit(0.0)) * -1.0
disc_amt = coalesce(col("DiscountAmount"), lit(0.0)) * -1.0
unit_price = coalesce(col("UnitPrice"), lit(0.0))

# Discounted unit price calculation rule
disc_unit_price = when(qty == 0, lit(0.0)).otherwise(coalesce(col("NetAmount"), lit(0.0)) / qty)

final_select_df = clean_result_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("documentId").alias("InvSrcId").cast("string"),
    col("id").alias("InvLnSrcId").cast("string"),
    lit('').alias("SOLnSrcId").cast("string"),
    
    # Structural cleanups with coalesce
    coalesce(col("locationId"), lit("")).alias("WHSrcId").cast("string"),
    coalesce(col("itemid"), lit("")).alias("ItemSrcId").cast("string"),
    coalesce(col("sequence"), lit("")).alias("InvLnKey").cast("string"),
    
    # Conditional type mappings
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED', 'ADM'), lit("Charge"))
        .otherwise(lit("CR")).alias("InvLnType").cast("string"),
        
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED'), lit("Freight"))
        .when(col("lineObjectNumber") == 'ADM', lit("AddlChrg"))
        .otherwise(lit("Credit Memo")).alias("InvLnTypeDesc").cast("string"),
    
    # Date normalizations
    when(col("shipmentDate") == '0001-01-01', lit(None)).otherwise(col("shipmentDate")).alias("ShipDt").cast("timestamp"),
    
    # Item definitions
    coalesce(col("description"), lit("")).alias("ItemDesc").cast("string"),
    when(col("lineObjectNumber").isin('FREIGHT OTHER GOODS', 'FREIGHT', 'FREIGHT NOT COVERED'), lit("Freight"))
        .when(col("lineObjectNumber") == 'ADM', lit("AddlChrg"))
        .otherwise(lit("Item")).alias("ItemType").cast("string"),
        
    coalesce(col("UnitOfMeasureID"), lit("")).alias("UOMSrcId").cast("string"),
    coalesce(col("unitofMeasureCode"), lit("")).alias("UOM").cast("string"),
    
    # Currency and conversion indicators
    lit("INR").alias("CustCurr").cast("string"),
    lit("INR").alias("LocCurr").cast("string"),
    lit(1.0).alias("ExRtLCCC").cast("double"),
    lit(1.0).alias("ExRtCCLC").cast("double"),
    lit(1.0).alias("ExRtLCIN").cast("double"),
    
    # Financial metrics maps
    inv_qty.alias("InvLnQty").cast("double"),
    (unit_price * lit(0.75)).alias("UnitCostCC").cast("double"),
    (unit_price * lit(0.75)).alias("UnitCostLC").cast("double"),
    (unit_price * lit(0.75)).alias("UnitCostIN").cast("double"),
    
    unit_price.alias("UnitPriceCC").cast("double"),
    unit_price.alias("UnitPriceLC").cast("double"),
    unit_price.alias("UnitPriceIN").cast("double"),
    
    disc_unit_price.alias("DiscUnitPriceCC").cast("double"),
    disc_unit_price.alias("DiscUnitPriceLC").cast("double"),
    disc_unit_price.alias("DiscUnitPriceIN").cast("double"),
    
    (net_amt * lit(0.75)).alias("InvLnCostCC").cast("double"),
    (net_amt * lit(0.75)).alias("InvLnCostLC").cast("double"),
    (net_amt * lit(0.75)).alias("InvLnCostIN").cast("double"),
    
    net_amt.alias("InvLnValCC").cast("double"),
    net_amt.alias("InvLnValLC").cast("double"),
    net_amt.alias("InvLnValIN").cast("double"),
    
    coalesce(col("DiscountPercent"), lit(0.0)).alias("InvLnDiscPerc").cast("double"),
    disc_amt.alias("InvLnDiscValCC").cast("double"),
    disc_amt.alias("InvLnDiscValLC").cast("double"),
    disc_amt.alias("InvLnDiscValIN").cast("double"),
    
    coalesce(col("TaxPercent"), lit(0.0)).alias("InvLnTaxPerc").cast("double"),
    (coalesce(col("NetTaxAmount"), lit(0.0)) * lit(-1.0)).alias("InvLnTaxValCC").cast("double"),
    (coalesce(col("NetTaxAmount"), lit(0.0)) * lit(-1.0)).alias("InvLnTaxValLC").cast("double"),
    (coalesce(col("NetTaxAmount"), lit(0.0)) * lit(-1.0)).alias("InvLnTaxValIN").cast("double"),
    
    # Metadata tracking fields
    lit(None).alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)
display(final_select_df.limit(10))

In [0]:
# ==========================================
# 5. HIGH-PERFORMANCE MULTI-TABLE CHAINED JOIN
# ==========================================
f_df = final_select_df.alias("f")
inv_df = spark.read.table(f"erp_lakehouse.{SILVER_SCHEMA}.sl_invoiceheader").alias("inv")
item_df = spark.read.table(f"erp_lakehouse.{SILVER_SCHEMA}.sl_item").alias("item")
prod_df = spark.read.table(f"erp_lakehouse.{SILVER_SCHEMA}.sl_productline").alias("prod")

combined_joins_df = f_df \
    .join(
        inv_df,
        (col("f.CompanyKey") == col("inv.CompanyKey")) & 
        (col("f.BrandKey") == col("inv.BrandKey")) & 
        (col("f.InvSrcId") == col("inv.InvSrcId")),
        "left"
    ) \
    .join(
        item_df,
        (col("f.CompanyKey") == col("item.CompanyKey")) & 
        (col("f.BrandKey") == col("item.BrandKey")) & 
        (col("f.ItemSrcId") == col("item.ItemSrcID")), 
        "left"
    ) \
    .join(
        prod_df,
        (col("f.CompanyKey") == col("prod.CompanyKey")) & 
        (col("f.BrandKey") == col("prod.BrandKey")) & 
        (col("item.ProductLnSrcId") == col("prod.ProductLnSrcId")),
        "left"
    )



In [0]:
# ==========================================
# 6. FINAL PROJECTION WITH SAFE NULL HANDLING
# ==========================================
Credits_final_df = combined_joins_df.select(
    # Core attributes from processed final data
    col("f.*"),
    
    # Left-table null handling enrichments from Invoice Header
    coalesce(col("inv.ExDt"), lit(None).cast("timestamp")).alias("ExDt"),
    coalesce(col("inv.SOKey"), lit("")).alias("SOSrcId").cast("string"),
    coalesce(col("inv.InvKey"), lit("")).alias("InvKey").cast("string"),
    coalesce(col("inv.InvStatus"), lit("Draft")).alias("InvLnStatus").cast("string"),
    
    # Left-table null handling enrichments from Item Master
    coalesce(col("item.ProductLnKey"), lit("")).alias("ProductLnKey").cast("string"),
    coalesce(col("item.ItemKey"), lit("")).alias("ItemKey").cast("string"),
    coalesce(col("item.ProductLnSrcId"), lit("")).alias("ProductLnSrcId").cast("string"),
    
    # Left-table null handling enrichments from Product Line Master
    coalesce(col("prod.ProductLnDesc"), lit("")).alias("ProductLnDesc").cast("string"),
    coalesce(col("prod.ProductLnType"), lit("")).alias("ProductLnType").cast("string")
)

# Output evaluation verification
display(Credits_final_df.limit(10))
print("Credit Memo Lines processing executed successfully! 🎉")


## combining both credits and invoices

In [0]:
invoiceline_df = Invoices_final_df.unionByName(Credits_final_df)

In [0]:


# Create table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.sl_invoiceline (
    CompanyKey string ,
    BrandKey string ,
    InvSrcId string,
    InvLnSrcId string,
    SOSrcId string,
    SOLnSrcId string ,
    WHSrcId string,
    ProductLnSrcId string,
    ItemSrcId string,
    InvKey string,
    InvLnKey string,
    InvLnStatus string,
    InvLnType string ,
    InvLnTypeDesc string ,
    ShipDt timestamp,
    ProductLnKey string,
    ProductLnDesc string,
    ProductLnType string,
    ItemKey string,
    ItemDesc string,
    ItemType string ,
    UOMSrcId string,
    UOM string,
    ExDt timestamp,
    CustCurr string ,
    LocCurr string ,
    ExRtLCCC double ,
    ExRtCCLC double ,
    ExRtLCIN double ,
    InvLnQty double,
    UnitCostCC double ,
    UnitCostLC double ,
    UnitCostIN double ,
    UnitPriceCC double,
    UnitPriceLC double,
    UnitPriceIN double,
    DiscUnitPriceCC double,
    DiscUnitPriceLC double,
    DiscUnitPriceIN double,
    InvLnCostCC double ,
    InvLnCostLC double ,
    InvLnCostIN double ,
    InvLnValCC double,
    InvLnValLC double,
    InvLnValIN double,
    InvLnDiscPerc double,
    InvLnDiscValCC double,
    InvLnDiscValLC double,
    InvLnDiscValIN double,
    InvLnTaxPerc double,
    InvLnTaxValCC double,
    InvLnTaxValLC double,
    InvLnTaxValIN double,
    SourceUpdatedTime timestamp,
    SysCreatedTime timestamp,
    RecordStatus string 
)
USING DELTA
LOCATION '{SILVER_PATH}'
""")

In [0]:
%sql
-- For the Invoice Line table
ALTER TABLE erp_lakehouse.silver.sl_invoiceline 
SET TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
);

In [0]:
# Create temp view for Credits_df
invoiceline_df.createOrReplaceTempView("Invoices_temp_view")

spark.sql("""
MERGE INTO erp_lakehouse.silver.sl_invoiceline AS tgt
USING Invoices_temp_view AS src
ON tgt.CompanyKey = src.CompanyKey
   AND tgt.BrandKey = src.BrandKey
   AND tgt.InvSrcId = src.InvSrcId
   AND tgt.InvLnSrcId = src.InvLnSrcId
WHEN MATCHED THEN
  UPDATE SET
    tgt.InvLnType = src.InvLnType,
    tgt.InvLnTypeDesc = src.InvLnTypeDesc,
    tgt.ShipDt = src.ShipDt,
    tgt.ItemDesc = src.ItemDesc,
    tgt.ItemType = src.ItemType,
    tgt.InvLnQty = src.InvLnQty,
    tgt.UnitCostCC = src.UnitCostCC,
    tgt.UnitCostLC = src.UnitCostLC,
    tgt.UnitCostIN = src.UnitCostIN,
    tgt.UnitPriceCC = src.UnitPriceCC,
    tgt.UnitPriceLC = src.UnitPriceLC,
    tgt.UnitPriceIN = src.UnitPriceIN,
    tgt.DiscUnitPriceCC = src.DiscUnitPriceCC,
    tgt.DiscUnitPriceLC = src.DiscUnitPriceLC,
    tgt.DiscUnitPriceIN = src.DiscUnitPriceIN,
    tgt.InvLnCostCC = src.InvLnCostCC,
    tgt.InvLnCostLC = src.InvLnCostLC,
    tgt.InvLnCostIN = src.InvLnCostIN,
    tgt.InvLnValCC = src.InvLnValCC,
    tgt.InvLnValLC = src.InvLnValLC,
    tgt.InvLnValIN = src.InvLnValIN,
    tgt.InvLnDiscPerc = src.InvLnDiscPerc,
    tgt.InvLnDiscValCC = src.InvLnDiscValCC,
    tgt.InvLnDiscValLC = src.InvLnDiscValLC,
    tgt.InvLnDiscValIN = src.InvLnDiscValIN,
    tgt.InvLnTaxPerc = src.InvLnTaxPerc,
    tgt.InvLnTaxValCC = src.InvLnTaxValCC,
    tgt.InvLnTaxValLC = src.InvLnTaxValLC,
    tgt.InvLnTaxValIN = src.InvLnTaxValIN,
    tgt.SourceUpdatedTime = src.SourceUpdatedTime,
    tgt.InvLnStatus = src.InvLnStatus,
    tgt.ProductLnKey = src.ProductLnKey,
    tgt.ItemKey = src.ItemKey,
    tgt.ProductLnDesc = src.ProductLnDesc,
    tgt.ProductLnType = src.ProductLnType
WHEN NOT MATCHED THEN
  INSERT *
""")

processed_df.unpersist()
print("Pipeline executed successfully and cleanly.")

In [0]:
%skip
OPTIMIZE erp_lakehouse.silver.sl_invoiceline;

In [0]:
# Enable unsafe vacuuming if you want to clear files immediately (< 7 days)
spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [0]:
# Run vacuum
spark.sql("VACUUM erp_lakehouse.silver.sl_invoiceline RETAIN 0 HOURS")